# Layers

> Potentially helpful layers for your models

In [ ]:
#| default_exp layers

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import torch, numpy as np, torch.nn.functional as F

from typing import Optional
from torch import nn
from torch import Tensor
from torch.fft import fft
from wavefm.augmentations import create_patch, mask_patches_simple, jitter_augmentation, shuffle_dim, reverse_sequence, channel_masking
from wavefm.signal import stft
from rotary_embedding_torch import RotaryEmbedding
from torch.nn.attention import SDPBackend, sdpa_kernel

## Miscellaneous

In [ ]:
#| export
class Identity(nn.Module):
    def __init__(self):
        super().__init__()
    def forward(self, x, **kwargs):
        return x
    
class Transpose(nn.Module):
    def __init__(self, *dims, contiguous=False): 
        super().__init__()
        self.dims, self.contiguous = dims, contiguous
    def forward(self, x):        
        if self.contiguous: return x.transpose(*self.dims).contiguous()
        else: return x.transpose(*self.dims)

def get_activation_fn(activation):
    if callable(activation): return activation()
    elif activation.lower() == "relu": return nn.ReLU()
    elif activation.lower() == "gelu": return nn.GELU()
    raise ValueError(f'{activation} is not available. You can use "relu", "gelu", or a callable')

class MovingAverage(nn.Module):
    "Moving average block to highlight the trend of time series"

    def __init__(self,
         kernel_size:int,  # the size of the window
         ):
        super().__init__()
        padding_left = (kernel_size - 1) // 2
        padding_right = kernel_size - padding_left - 1
        self.padding = torch.nn.ReplicationPad1d((padding_left, padding_right))
        self.avg = nn.AvgPool1d(kernel_size=kernel_size, stride=1)

    def forward(self, x:Tensor):
        """ 
        Args:
            x: torch.Tensor shape: [bs x seq_len x features]
        """
        return self.avg(self.padding(x))
    
class SeriesDecomposition(nn.Module):
    "Series decomposition block"

    def __init__(self,
         kernel_size:int,  # the size of the window
         ):
        super().__init__()
        self.moving_avg = MovingAverage(kernel_size)

    def forward(self, x:Tensor):
        """ Args:
            x: torch.Tensor shape: [bs x seq_len x features]
        """
        moving_mean = self.moving_avg(x)
        residual = x - moving_mean
        return residual, moving_mean

## Linear Layers for Patches

In [ ]:
#| export
class PatchEncoder3D(nn.Module):
    """
    3d convolution for patched time series data, broken into frames
    """

    def __init__(
        self,
        c_in,
        patch_size, # patch len per frame
        n_patches, # number of patches per frame
        tubelet_size, # how many frames to process at a time
        d_model,
    ):
        super().__init__()
        self.patch_size = patch_size
        self.n_patches = n_patches
        self.tubelet_size = tubelet_size

        self.proj = nn.Conv3d(
            in_channels=c_in,
            out_channels=d_model,
            kernel_size=(tubelet_size, n_patches, patch_size),
            stride=(tubelet_size, n_patches, patch_size),
        )

    def forward(self, x, **kwargs):
        B, C, T, H, W = x.shape
        x = self.proj(x).flatten(start_dim=2).transpose(1, 2) # [bs x n tubelets x d_model]
        return x

In [ ]:
#| notest
m = PatchEncoder3D(c_in=7, patch_size = 3, n_patches = 12, tubelet_size=2, d_model = 512)

x = torch.randn(4, 7, 10, 12, 3)
m(x).shape


torch.Size([4, 5, 512])

## Positional Encoding Layers

In [ ]:
#| export
class PositionalEncoding(nn.Module):
    def __init__(self, 
                 num_patch, # number of patches of time series or stft in input
                 d_model, # dimension of patch embeddings
                 #dropout=0.1 # dropout value
                 ):
        super().__init__()
        self.num_patch = num_patch
        self.d_model = d_model
        
        # Positional encoding - learned
        self.W_pos =  nn.Parameter(torch.empty((num_patch, d_model)))
        nn.init.uniform_(self.W_pos, -0.02, 0.02)
        self.scale_factor = nn.Parameter(torch.ones(1))
        #self.dropout = nn.Dropout(p=dropout)

    def forward(self, x):
        """
        input: x: [bs * nvars x num_patch x d_model]
        returns: x: [bs * nvars x num_patch x d_model]
        """
        if x.is_nested:
            out_tensors = []
            for x_i in x:
                # Only use the portion of W_pos that matches this sequence length
                seq_len = x_i.size(0) # get number of patches
                x_i = x_i + self.scale_factor * self.W_pos[:seq_len]
                out_tensors.append(x_i)
            x = torch.nested.as_nested_tensor(out_tensors, layout=torch.jagged)
        else:
            x = x + self.scale_factor * self.W_pos
        return x

In [ ]:
#| notest
d_model = 512
n_heads = 8
batch_size = 2
n_vars = 7
max_len = 100

# Create sequences of different lengths
seq_lens = torch.randint(50, max_len, (batch_size,))

# Create input tensors with different sequence lengths
x_list = [torch.randn(length, d_model) for length in seq_lens]
x_nested = torch.nested.as_nested_tensor(x_list, layout=torch.jagged)

p = PositionalEncoding(num_patch=max_len, d_model=d_model)
out = p(x_nested)

In [ ]:
#| export
class tAPE(nn.Module):
    """
    time Absolute Position Encoding
    Adapted from tsai
    """

    def __init__(self, 
        d_model:int, # the embedding dimension
        seq_len:int, # the max. length of the incoming sequence or num patches
        ):
        super().__init__()
        self.scale_factor = nn.Parameter(torch.ones(1)) # learnable scale factor
        W_pos = torch.zeros(seq_len, d_model)  # positional encoding
        position = torch.arange(0, seq_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))

        W_pos[:, 0::2] = torch.sin((position * div_term)*(d_model/seq_len)) # this is the difference between normal PE and tAPE, scaling (d_model/seq_len)
        W_pos[:, 1::2] = torch.cos((position * div_term)*(d_model/seq_len))
        self.register_buffer('W_pos', W_pos)  # this stores the variable in the state_dict (used for non-trainable variables)
        #self.W_pos = self.scale_factor * self.W_pos.unsqueeze(0)
        

        #self.dropout = nn.Dropout(p=dropout)

    def forward(self, x): # [batch size, sequence length, embed dim]
        if x.is_nested:
            out_tensors = []
            for x_i in x:
                seq_len = x_i.size(0) # get number of patches
                pos_ = self.W_pos[:seq_len] * self.scale_factor
                x_i = x_i + pos_ #* self.scale_factor
                out_tensors.append(x_i)
            x = torch.nested.as_nested_tensor(out_tensors, layout=torch.jagged)
        else:
            x = x + self.W_pos * self.scale_factor
        return x
    


In [ ]:
#| notest
d_model = 768
batch_size = 2
n_vars = 7
max_len = 14400

# Create sequences of different lengths
seq_lens = torch.randint(10000, max_len, (batch_size,))

# Create input tensors with different sequence lengths
x_list = [torch.randn(length, d_model) for length in seq_lens]
x_nested = torch.nested.as_nested_tensor(x_list, layout=torch.jagged)

p = tAPE(seq_len=max_len, d_model=d_model)
out = p(x_nested)
out.shape

Any nans: tensor(False)
Any nans in x: tensor(False)
Any nans: tensor(False)
Any nans in x: tensor(False)


torch.Size([2, j15, 768])

## Mask and Augmentation Layers

In [ ]:
#| export
class Mask(nn.Module):
    def __init__(self, mask_type, mask_ratio, return_mask=True):
        super().__init__()
        assert mask_type in ['patch', 'jitter_zero']
        self.mask_type = mask_type
        self.mask_ratio = mask_ratio
        self.return_mask = return_mask
    
    def forward(self, x):
        if self.mask_type == 'jitter_zero':
            # mask is a number with the number of masks applied in this function
            x_masked, mask = jitter_augmentation(x, mask_ratio=(self.mask_ratio/2), jitter_ratio=(self.mask_ratio/2))
        else:
            # padding mask is currently not implemented here.. tbh not sure if its needed
            x_masked, mask = mask_patches_simple(x, mask_ratio=self.mask_ratio)
        if self.return_mask:
            return x_masked, mask
        else:
            return x_masked


In [ ]:
#| notest
torch.manual_seed(125)
m = Mask(mask_type='jitter_zero', mask_ratio=0.5)
x = torch.randn((9))

m(x), m(x)

In [ ]:
import torch
x = [0,1,2,3]
[x[i] for i in torch.randperm(len(x))]

[3, 0, 2, 1]

In [ ]:
#| export
class PatchAugmentations(nn.Module):
    def __init__(self, augmentations=['patch_mask', 'jitter_zero_mask', 'reverse_sequence', 'shuffle_channels', 'channel_masking'], patch_mask_ratio=0., jitter_zero_mask_ratio=0.):
        super().__init__()
        self.augmentations = augmentations
        self.patch_mask_ratio = patch_mask_ratio
        self.jitter_zero_mask_ratio = jitter_zero_mask_ratio 
    def forward(self, x):
        if 'patch_mask' in self.augmentations and 'jitter_zero_mask' in self.augmentations:
            c = ['patch_mask', 'jitter_zero_mask']
            mask_choice = c[torch.randperm(2)[0].item()] #np.random.choice(['patch_mask', 'jitter_zero_mask'], replace=True)
        else:
            mask_choice = None
        augs = self.augmentations.copy()
        augs = [augs[i] for i in torch.randperm(len(augs)).tolist()]
        for augmentation in augs:
            if augmentation == 'jitter_zero_mask' and (mask_choice == 'jitter_zero_mask' or mask_choice is None):
                if isinstance(self.jitter_zero_mask_ratio, tuple) or isinstance(self.jitter_zero_mask_ratio, list):
                    mask_ratio = self.jitter_zero_mask_ratio[0] + torch.rand(1).item() * (self.jitter_zero_mask_ratio[1] - self.jitter_zero_mask_ratio[0])
                else:
                    mask_ratio = self.jitter_zero_mask_ratio
                # mask is a number with the number of masks applied in this function
                x = jitter_augmentation(x, mask_ratio=mask_ratio, jitter_ratio=mask_ratio)
            if augmentation == 'patch_mask' and (mask_choice == 'patch_mask' or mask_choice is None):
                # padding mask is currently not implemented here.. tbh not sure if its needed
                if isinstance(self.patch_mask_ratio, tuple) or isinstance(self.patch_mask_ratio, list):
                    mask_ratio = self.patch_mask_ratio[0] + torch.rand(1).item() * (self.patch_mask_ratio[1] - self.patch_mask_ratio[0])
                else:
                    mask_ratio = self.patch_mask_ratio
                x = mask_patches_simple(x, mask_ratio=mask_ratio)
            if augmentation == 'channel_masking':
                x = channel_masking(x, dim=2, p=0.5)
            if augmentation == 'shuffle_channels':
                x = shuffle_dim(x, dim=2, p=0.5)
            if augmentation == 'reverse_sequence':
                x = reverse_sequence(x, seq_dim=(-1,), p=0.5)
        return x

In [ ]:
#| notest
x = torch.randn(4,3600,7,750)

s=PatchAugmentations(augmentations=[], patch_mask_ratio=0.1, jitter_zero_mask_ratio=0.1)
s(x).shape
torch.equal(x, s(x))

True

In [ ]:
#| export
class EmbeddingAugmentations(nn.Module):
    def __init__(self, augmentations=['shuffle_dims', 'jitter_zero_mask', 'patch_mask'], dims_to_shuffle = [1,2,3], patch_mask_ratio=0., jitter_zero_mask_ratio=0.):
        super().__init__()
        assert set(augmentations) - set(['shuffle_dims', 'jitter_zero_mask', 'patch_mask']) == set(), f"Augmentations must be in {set(['shuffle_dims', 'jitter_zero_mask', 'patch_mask'])}."
        if 'patch_mask' in augmentations:
            assert patch_mask_ratio >= 0.
        if 'jitter_zero_mask' in augmentations:
            assert jitter_zero_mask_ratio >= 0.
        self.augmentations = augmentations
        self.patch_mask_ratio = patch_mask_ratio
        self.jitter_zero_mask_ratio = jitter_zero_mask_ratio
        self.dims_to_shuffle = dims_to_shuffle
    
    def forward(self, x):
        """
        Input is an embedding: 
        x: [bs x n channels x d model x n patches]
        dims correspond to 1 = channels 2 = n patches 3 = d model
        returns: [bs x n channels x d model x n patches]
        """
        x = x.permute(0,1,3,2)
        if 'patch_mask' in self.augmentations and 'jitter_zero_mask' in self.augmentations:
            c = ['patch_mask', 'jitter_zero_mask']
            mask_choice = c[torch.randperm(2)[0].item()]
        else:
            mask_choice = None
        augs = self.augmentations
        augs = [augs[i] for i in torch.randperm(len(augs)).tolist()]
        for augmentation in augs:
            if augmentation == 'jitter_zero_mask' and (mask_choice == 'jitter_zero_mask' or mask_choice is None):
                # mask is a number with the number of masks applied in this function
                x, _ = jitter_augmentation(x, mask_ratio=(self.jitter_zero_mask_ratio/2), jitter_ratio=(self.jitter_zero_mask_ratio/2))
            if augmentation == 'patch_mask' and (mask_choice == 'patch_mask' or mask_choice is None):
                # padding mask is currently not implemented here.. tbh not sure if its needed
                x, _ = mask_patches_simple(x, mask_ratio=self.patch_mask_ratio)
            if augmentation == 'shuffle_dims':
                # shuffle channels, patches, features
                dim_shuffle_order = self.dims_to_shuffle
                dim_shuffle_order = [dim_shuffle_order[i] for i in torch.randperm(len(dim_shuffle_order)).tolist()]
                for dim in dim_shuffle_order:
                    x = shuffle_dim(x, dim=dim, p=0.5)
        x = x.permute(0,1,3,2)
        return x

In [ ]:
#| notest
x = torch.randn(4,7,512,3600)

s = EmbeddingAugmentations(augmentations=['jitter_zero_mask'], dims_to_shuffle=[1], patch_mask_ratio=0.1, jitter_zero_mask_ratio=0.1)
s(x).shape

torch.Size([4, 7, 512, 3600])

## Patch and Fourier Layers

In [ ]:
#| export
class Patch(nn.Module):
    def __init__(self, patch_len, stride):
        super().__init__()
        self.patch_len = patch_len
        self.stride = stride
    
    def forward(self, x, constant_pad=False, constant_pad_value=0):
        x = create_patch(x, patch_len=self.patch_len, stride=self.stride, constant_pad=constant_pad, constant_pad_value=constant_pad_value)
        return x
    
#| export
class Patch3d(nn.Module):
    def __init__(self, frame_len, frame_stride, patch_len, patch_stride):
        super().__init__()
        self.frame_len = frame_len
        self.frame_stride = frame_stride
        self.patch_len = patch_len
        self.patch_stride = patch_stride
    
    def forward(self, x, constant_pad=False, constant_pad_value=0):
        x = create_patch3d(x, patch_len=self.patch_len, patch_stride=self.patch_stride, frame_len=self.frame_len, frame_stride=self.frame_stride, constant_pad=constant_pad, constant_pad_value=constant_pad_value)
        return x

In [ ]:
#| export 
class STFT(nn.Module):
    def __init__(self, 
                 n_fft, 
                 win_length,
                 hop_length, 
                 stft_norm, 
                 decibel_scale, 
                 channel_stft_means=None, 
                 channel_stft_stds=None, 
                 pad_win_length_to_nfft=True, 
                 pad_mode='reflect', 
                 center=False, 
                 return_complex=True
                 ):
        super().__init__()
        self.n_fft = n_fft
        self.win_length = win_length
        self.hop_length = hop_length
        self.pad_mode = pad_mode
        self.pad_win_length_to_nfft = pad_win_length_to_nfft
        self.center = center
        self.return_complex = return_complex
        self.stft_norm = stft_norm
        self.decibel_scale = decibel_scale
        self.channel_stft_means = channel_stft_means
        self.channel_stft_stds = channel_stft_stds

    def forward(self, x):
        """
        x: [bs x n_vars x max_seq_len]
        out: [bs x n_vars x n_fft // 2 + 1 x stft_len]
        """
        x_fft = stft(x, 
                     n_fft=self.n_fft, 
                     win_length=self.win_length,
                     normalized=self.stft_norm,
                     pad_win_length_to_nfft=self.pad_win_length_to_nfft,
                     decibel_scale=self.decibel_scale, 
                     channel_stft_means=self.channel_stft_means, 
                     channel_stft_stds=self.channel_stft_stds, 
                     hop_length=self.hop_length, 
                     pad_mode=self.pad_mode, 
                     center=self.center, 
                     return_complex=self.return_complex
                     )
        return x_fft


In [ ]:
#| export 
class FFT(nn.Module):
    def __init__(self, 
                 dim=-1, # dimension to calculate fft over
                 norm='backward' #  "forward" - normalize by 1/n, "backward" - no normalization, "ortho" - normalize by 1/sqrt(n) (making the FFT orthonormal)
                 ):
        super().__init__()
        self.dim = dim
        self.norm = norm

    def forward(self, x):
        """
        This is equivalent to an stft with onesided set to False, hop length of 1, and win_length == n_fft.

        x: [bs x num_patch x n_vars x patch_len]
        out: [bs x num_patch x n_vars x patch_len(freq domain)]
        """
        if x.is_nested:
            x_list = list(x.unbind())
            for i, x_i in enumerate(x_list):
                x_list[i] = fft(x_i, dim=self.dim, norm=self.norm).abs()
            x_fft = torch.nested.as_nested_tensor(x_list, layout=torch.jagged)
        else:
            x_fft = fft(x, dim=self.dim, norm=self.norm).abs()
        return x_fft


In [ ]:
#| notest
patch_len = 750
d_model = 12
n_vars = 7
max_len = 100
bs = 3
# Create sequences of different lengths
seq_lens = torch.randint(50, max_len, (bs,))

# Create input tensors with different sequence lengths
x_list = [torch.randn(length, n_vars, patch_len) for length in seq_lens]
x_nested = torch.nested.as_nested_tensor(x_list, layout=torch.jagged)

f = FFT(dim=-1)
x = f(x_nested)
x.size(0), x.dim()

(3, 4)

## Reversible Instance Normalization

In [ ]:
#| export
class RevIN(nn.Module):
    def __init__(self, 
                 num_features: int, # the number of channels or features in the input
                 eps=1e-5, # added to avoid division by zero errors
                 dim_to_reduce=-1, # the dimension to reduce, 
                 affine=True # learning affine parameters bias and weight per channel
                 ):
        """
        
        """
        super().__init__()
        self.num_features = num_features
        self.eps = eps
        self.affine = affine
        self.dim_to_reduce = dim_to_reduce

        if self.affine:
            self.affine_weight = nn.Parameter(torch.ones(num_features,1))
            self.affine_bias = nn.Parameter(torch.zeros(num_features,1))

    def forward(self, x, mode:bool):
        """
        x: [bs x n_vars x max_seq_len]
        """
        if mode:
            return self._normalize(x)
        else:
            return self._denormalize(x)

    def _normalize(self, x):
        if x.is_nested:
            x_revin = []
            self.mean = []
            self.stdev = []
            for x_i in x:
                self.mean.append(torch.mean(x_i, dim=self.dim_to_reduce, keepdim=True).detach())
                self.stdev.append(torch.std(x_i, dim=self.dim_to_reduce, keepdim=True, unbiased=False).detach() + self.eps)
                x_i = x_i.sub(self.mean[-1])
                x_i = x_i.div(self.stdev[-1])
                if self.affine:
                    x_i = x_i.mul(self.affine_weight)
                    x_i = x_i.add(self.affine_bias)
                x_revin.append(x_i.transpose(0,1)) # first dim is jagged
            x = torch.nested.as_nested_tensor(x_revin, layout=torch.jagged)
            x = x.transpose(1,2)
        else:
            self.mean = torch.mean(x, dim=self.dim_to_reduce, keepdim=True).detach()
            self.stdev = torch.std(x, dim=self.dim_to_reduce, keepdim=True, unbiased=False).detach() + self.eps
            x = x.sub(self.mean)
            x = x.div(self.stdev)
            if self.affine:
                x = x.mul(self.affine_weight)
                x = x.add(self.affine_bias)
        return x

    def _denormalize(self, x):
        if x.is_nested:
            x_revin = []
            for i, x_i in enumerate(x):
                if self.affine:
                    x_i = x_i.sub(self.affine_bias)
                    x_i = x_i.div(self.affine_weight)
                x_i = x_i.mul(self.stdev[i])
                x_i = x_i.add(self.mean[i])
                x_revin.append(x_i.transpose(0,1))
            x = torch.nested.as_nested_tensor(x_revin, layout=torch.jagged)
            x = x.transpose(1,2)
        else:
            if self.affine:
                x = x.sub(self.affine_bias)
                x = x.div(self.affine_weight)
            x = x.mul(self.stdev)
            x = x.add(self.mean)
        return x

In [ ]:
#| notest
x = torch.randn(2,7,100)
revin = RevIN(7, dim_to_reduce=-1, affine=True)

x_norm = revin(x, mode=True)
x_denorm = revin(x_norm, mode=False)
x_norm.shape, x_denorm.shape

(torch.Size([2, 7, 100]), torch.Size([2, 7, 100]))

In [ ]:
#| notest
batch_size = 2
n_vars = 7
max_len = 100

# Create sequences of different lengths
seq_lens = torch.randint(50, max_len, (batch_size,))

# Create input tensors with different sequence lengths
x_list = [torch.randn(n_vars, length) for length in seq_lens]
x_nested = torch.nested.as_nested_tensor(x_list, layout=torch.jagged)
print(x_nested.shape)

revin = RevIN(n_vars, dim_to_reduce=-1, affine=True)
x_norm = revin(x_nested, mode=True)
print(x_norm.shape)
x_denorm = revin(x_norm, mode=False)
print(x_denorm.shape)


torch.Size([2, 7, j26])
torch.Size([2, 7, j27])
torch.Size([2, 7, j28])


## Masked Channel Tokens

In [ ]:
#| export
class LearnableMaskedChannelTokens(nn.Module):
    def __init__(self, missing_channel_indices, d_model):
        super().__init__()
        # Create learnable tokens for each channel and each potential patch position
        self.missing_channel_indices = missing_channel_indices
        # create learnable tokens for each missing channel, but not per patch position
        self.mask_token = nn.Parameter(torch.zeros(len(missing_channel_indices), d_model, 1))
        torch.nn.init.normal_(self.mask_token, std=.02)
        
    def forward(self, x):
        """
        x: Input tensor [bs x nvars x d_model x num_patch] 
        """
        if x.is_nested:
            x_list = list(x.unbind())
            for i, x_i in enumerate(x_list):
                t = x_i.clone()
                t[self.missing_channel_indices] = t[self.missing_channel_indices] + self.mask_token
                x_list[i] = t
            x = torch.nested.as_nested_tensor(x_list, layout=torch.jagged)
        else:
            x[:, self.missing_channel_indices] = x[:, self.missing_channel_indices] + self.mask_token
        return x


In [ ]:
#| notest
x = torch.randn(2,7,512,3600)
missing_channel_indices = [0,1]
learnable_mask_tokens = LearnableMaskedChannelTokens(missing_channel_indices, d_model=512)
learnable_mask_tokens(x).shape

seq_lens = torch.tensor([50, 100])

# Create input tensors with different sequence lengths
x_list = [torch.randn(n_vars,512, length) for length in seq_lens]
x_nested = torch.nested.as_nested_tensor(x_list, layout=torch.jagged)

out = learnable_mask_tokens(x_nested)
out.shape


torch.Size([2, 7, 512, j99])

## Inception

In [ ]:
#| export
class InceptionModule(nn.Module):
    """
    Inception module adapted from https://github.com/timeseriesAI/tsai/blob/main/tsai/models/InceptionTime.py
    """
    def __init__(self, 
                 in_channels: int,
                 bottleneck_channels: int = 32,
                 bottleneck = True,
                 kernel_size: int = 40,
                 groups: int = 1
                 ):
        super().__init__()
        kernel_size = [kernel_size // (2**i) for i in range(3)]
        kernel_size = [k if k % 2 != 0 else k - 1 for k in kernel_size]  # ensure odd ks
        bottleneck = bottleneck if in_channels > 1 else False
        assert bottleneck_channels % groups == 0, f"bottleneck_channels ({bottleneck_channels}) must be divisible by groups ({groups})"
        assert in_channels % groups == 0, f"in_channels ({in_channels}) must be divisible by groups ({groups})"
        self.bottleneck = nn.Conv1d(in_channels, bottleneck_channels, 1, bias=False, groups=groups) if bottleneck else nn.Identity()
        self.convs = nn.ModuleList([nn.Conv1d(bottleneck_channels if bottleneck else in_channels, bottleneck_channels, k, padding=k//2, bias=False, groups=groups) for k in kernel_size])
        self.maxconvpool = nn.Sequential(*[nn.MaxPool1d(3, stride=1, padding=1), nn.Conv1d(in_channels, bottleneck_channels, 1, bias=False, groups=groups)])
        self.bn = nn.BatchNorm1d(bottleneck_channels * 4)
        self.act = nn.ReLU()

    def forward(self, x):
        input_tensor = x
        x = self.bottleneck(input_tensor)
        x = torch.cat([l(x) for l in self.convs] + [self.maxconvpool(input_tensor)], dim=1)
        return self.act(self.bn(x))

class InceptionBlock(nn.Module):
    def __init__(self, in_channels, bottleneck_channels=32, residual=True, depth=6, groups=1, **kwargs):
        super().__init__()
        self.residual, self.depth = residual, depth
        self.inception, self.shortcut = nn.ModuleList(), nn.ModuleList()
        self.bottleneck_channels = bottleneck_channels
        self.groups = groups
        for d in range(depth):
            self.inception.append(InceptionModule(in_channels if d == 0 else bottleneck_channels * 4, bottleneck_channels, groups=groups, **kwargs))
            if self.residual and d % 3 == 2: 
                n_in, n_out = in_channels if d == 2 else bottleneck_channels * 4, bottleneck_channels * 4
                if n_in == n_out:
                    self.shortcut.append(nn.BatchNorm1d(n_in))
                else:
                    self.shortcut.append(nn.Sequential(nn.Conv1d(n_in, n_out, 1, bias=False, groups=groups), nn.BatchNorm1d(n_out)))
        self.act = nn.ReLU()
        
    def forward(self, x):
        res = x
        for d, l in enumerate(range(self.depth)):
            x = self.inception[d](x)
            if self.residual and d % 3 == 2: 
                y = self.shortcut[d//3](res)
                x = x.add(y)
                res = x = self.act(x)
        return x

## Attention

In [ ]:
#| export
class MLP(nn.Module):
    def __init__(
        self,
        in_features,
        hidden_features=None,
        out_features=None,
        act_layer=nn.GELU,
        drop=0.
    ):
        super().__init__()
        out_features = out_features or in_features
        hidden_features = hidden_features or in_features
        self.fc1 = nn.Linear(in_features, hidden_features)
        self.act = act_layer()
        self.fc2 = nn.Linear(hidden_features, out_features)
        self.drop = nn.Dropout(drop)

    def forward(self, x):
        x = self.fc1(x)
        x = self.act(x)
        x = self.drop(x)
        x = self.fc2(x)
        x = self.drop(x)
        return x

class MultiHeadAttention(nn.Module):
    def __init__(
        self,
        dim,
        num_heads=8,
        qkv_bias=False,
        qk_scale=None,
        attn_drop=0.,
        proj_drop=0.,
        rotary_pes=False,
        max_n_patches_rotary=14500
    ):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.scale = qk_scale or self.head_dim ** -0.5
        #self.qkv = nn.Linear(dim, dim * 3, bias=qkv_bias)
        self.W_Q = nn.Linear(dim, dim, bias=qkv_bias)
        self.W_K = nn.Linear(dim, dim, bias=qkv_bias)
        self.W_V = nn.Linear(dim, dim, bias=qkv_bias)
        self.attn_drop_prob = attn_drop
        #self.attn_drop = nn.Dropout(attn_drop)
        self.proj = nn.Linear(dim, dim)
        self.proj_drop = nn.Dropout(proj_drop)
        self.rotary_pes = rotary_pes
        if self.rotary_pes:
            self.rotary_embed = RotaryEmbedding(dim=self.head_dim,
                                                freqs_for="lang",
                                                theta=10000,
                                                learned_freq=False,
                                                seq_before_head_dim=False,
                                                use_xpos=False,
                                                cache_max_seq_len=max_n_patches_rotary # max n patches ? 
                                                )

    def forward(self, x, key=None, value=None, mask=None):
        """
        Attention
        This handles nested or regular tensors
        """
        if key is None:
            key = x
        if value is None:
            value = x
        # Split and reshape 
        q = self.W_Q(x).unflatten(-1, [self.num_heads, self.head_dim]).transpose(1, 2)
        k = self.W_K(key).unflatten(-1, [self.num_heads, self.head_dim]).transpose(1, 2)
        v = self.W_V(value).unflatten(-1, [self.num_heads, self.head_dim]).transpose(1, 2)
        if self.rotary_pes:
            if q.is_nested:
                q_nested = []
                k_nested = []
                for q_i, k_i in zip(q, k):
                    if not self.training:
                        q_i, k_i = self.rotary_embed.rotate_queries_with_cached_keys(q_i.unsqueeze(0), k_i.unsqueeze(0))
                    else:
                        q_i = self.rotary_embed.rotate_queries_or_keys(q_i.unsqueeze(0))
                        k_i = self.rotary_embed.rotate_queries_or_keys(k_i.unsqueeze(0))
                    q_nested.append(q_i.squeeze(0).transpose(0, 1))
                    k_nested.append(k_i.squeeze(0).transpose(0, 1))
                q = torch.nested.as_nested_tensor(q_nested, layout=torch.jagged)
                k = torch.nested.as_nested_tensor(k_nested, layout=torch.jagged)
                q = q.transpose(1, 2)
                k = k.transpose(1, 2)
            else:
                if not self.training:
                    q, k = self.rotary_embed.rotate_queries_with_cached_keys(q, k)
                else:
                    q = self.rotary_embed.rotate_queries_or_keys(q)
                    k = self.rotary_embed.rotate_queries_or_keys(k)
        #B, N, C = x.shape
        #qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, C // self.num_heads).permute(2, 0, 3, 1, 4)
        #q, k, v = qkv[0], qkv[1], qkv[2]
        attn_dropout = self.attn_drop_prob if self.training else 0.0
        with sdpa_kernel([SDPBackend.FLASH_ATTENTION, SDPBackend.EFFICIENT_ATTENTION, SDPBackend.MATH], set_priority=True):
             x = F.scaled_dot_product_attention(q, k, v, dropout_p=attn_dropout, attn_mask=mask, is_causal=False, scale=self.scale)
        # attn = (q @ k.transpose(-2, -1)) * self.scale  # [B, num_heads, D, D]
        # attn = attn.softmax(dim=-1)
        # attn = self.attn_drop(attn)
        #x = (attn @ v)

        x = x.transpose(1, 2).flatten(-2)
        #x = x.transpose(1, 2).reshape(B, N, C)
        x = self.proj(x)
        x = self.proj_drop(x)
        return x

In [ ]:
#| export
class ScaledDotProductAttention(nn.Module):
    """Scaled Dot-Product Attention module (Attention is all you need by Vaswani et al., 2017) with optional residual attention from previous layer
    (Realformer: Transformer likes residual attention by He et al, 2020) and locality self sttention (Vision Transformer for Small-Size Datasets
    by Lee et al, 2021)"""

    def __init__(self, d_model, n_heads, attn_dropout=0., res_attention=False, lsa=False):
        super().__init__()
        self.attn_dropout = nn.Dropout(attn_dropout)
        self.res_attention = res_attention
        head_dim = d_model // n_heads
        self.scale = nn.Parameter(torch.tensor(head_dim ** -0.5), requires_grad=lsa)
        self.lsa = lsa

    def forward(self, q:Tensor, k:Tensor, v:Tensor, prev:Optional[Tensor]=None, key_padding_mask:Optional[Tensor]=None, attn_mask:Optional[Tensor]=None):
        '''
        Input shape:
            q               : [bs x n_heads x max_q_len x d_k]
            k               : [bs x n_heads x d_k x seq_len]
            v               : [bs x n_heads x seq_len x d_v]
            prev            : [bs x n_heads x q_len x seq_len]
            key_padding_mask: [bs x seq_len]
            attn_mask       : [1 x seq_len x seq_len]
        Output shape:
            output:  [bs x n_heads x q_len x d_v]
            attn   : [bs x n_heads x q_len x seq_len]
            scores : [bs x n_heads x q_len x seq_len]
        '''

        # Scaled MatMul (q, k) - similarity scores for all pairs of positions in an input sequence
        ## this is the b ottleneck in your code
        attn_scores = torch.matmul(q, k) * self.scale      # attn_scores : [bs x n_heads x max_q_len x q_len]
        # Add pre-softmax attention scores from the previous layer (optional)
        if prev is not None: attn_scores = attn_scores + prev

        ## if implementing attention or key padding mask - https://github.com/pytorch/pytorch/issues/41508#issuecomment-1723119580
        # Attention mask (optional)
        if attn_mask is not None:                                     # attn_mask with shape [q_len x seq_len] - only used when q_len == seq_len
            if attn_mask.dtype == torch.bool:
                attn_scores.masked_fill_(attn_mask, -np.inf)
            else:
                attn_scores += attn_mask

        # Key padding mask (optional)
        if key_padding_mask is not None:# and torch.any(key_padding_mask): - not sure if this is needed                         # mask with shape [bs x q_len] (only when max_w_len == q_len)
            attn_scores.masked_fill_(key_padding_mask.unsqueeze(1).unsqueeze(2).to(torch.bool), -np.inf)

        # normalize the attention weights
        # this is another bottleneck (softmax)
        attn_weights = F.softmax(attn_scores, dim=-1)                 # attn_weights   : [bs x n_heads x max_q_len x q_len]
        if attn_mask is not None and attn_mask.dtype == torch.bool:
            attn_weights = attn_weights.masked_fill(attn_mask, 0.0)
       
        if key_padding_mask is not None:# and torch.any(key_padding_mask):
            attn_weights = attn_weights.masked_fill(key_padding_mask.unsqueeze(1).unsqueeze(2).to(torch.bool), 0.0)
       
        attn_weights = self.attn_dropout(attn_weights)
        # compute the new values given the attention weights
        output = torch.matmul(attn_weights, v)                        # output: [bs x n_heads x max_q_len x d_v]
        if self.res_attention: return output, attn_weights, attn_scores
        else: return output, attn_weights


In [ ]:
#| export
class MultiheadAttentionCustom(nn.Module):
    def __init__(self, d_model, n_heads, d_k=None, d_v=None, res_attention=False, attn_dropout=0., proj_dropout=0., qkv_bias=True, lsa=False):
        """Multi Head Attention Layer
        Input shape:
            Q:       [batch_size (bs) x max_q_len x d_model]
            K, V:    [batch_size (bs) x q_len x d_model]
            mask:    [q_len x q_len]
        """
        super().__init__()
        d_k = d_model // n_heads if d_k is None else d_k
        d_v = d_model // n_heads if d_v is None else d_v

        self.n_heads, self.d_k, self.d_v = n_heads, d_k, d_v

        self.W_Q = nn.Linear(d_model, d_k * n_heads, bias=qkv_bias)
        self.W_K = nn.Linear(d_model, d_k * n_heads, bias=qkv_bias)
        self.W_V = nn.Linear(d_model, d_v * n_heads, bias=qkv_bias)

        # Scaled Dot-Product Attention (multiple heads)
        self.res_attention = res_attention
        self.sdp_attn = ScaledDotProductAttention(d_model, n_heads, attn_dropout=attn_dropout, res_attention=self.res_attention, lsa=lsa)

        # Poject output
        self.to_out = nn.Sequential(nn.Linear(n_heads * d_v, d_model), nn.Dropout(proj_dropout))


    def forward(self, Q:Tensor, K:Optional[Tensor]=None, V:Optional[Tensor]=None, prev:Optional[Tensor]=None,
                key_padding_mask:Optional[Tensor]=None, attn_mask:Optional[Tensor]=None):

        bs = Q.size(0)
        if K is None: K = Q
        if V is None: V = Q

        # Linear (+ split in multiple heads)
        q_s = self.W_Q(Q).view(bs, -1, self.n_heads, self.d_k).transpose(1,2)       # q_s    : [bs x n_heads x max_q_len x d_k]
        k_s = self.W_K(K).view(bs, -1, self.n_heads, self.d_k).permute(0,2,3,1)     # k_s    : [bs x n_heads x d_k x q_len] - transpose(1,2) + transpose(2,3)
        v_s = self.W_V(V).view(bs, -1, self.n_heads, self.d_v).transpose(1,2)       # v_s    : [bs x n_heads x q_len x d_v]
        # Apply Scaled Dot-Product Attention (multiple heads)
        if self.res_attention:
            output, attn_weights, attn_scores = self.sdp_attn(q_s, k_s, v_s, prev=prev, key_padding_mask=key_padding_mask, attn_mask=attn_mask)
        else:
            output, attn_weights = self.sdp_attn(q_s, k_s, v_s, key_padding_mask=key_padding_mask, attn_mask=attn_mask)
        # output: [bs x n_heads x q_len x d_v], attn: [bs x n_heads x q_len x q_len], scores: [bs x n_heads x max_q_len x q_len]

        # back to the original inputs dimensions
        output = output.transpose(1, 2).contiguous().view(bs, -1, self.n_heads * self.d_v) # output: [bs x q_len x n_heads * d_v]
        output = self.to_out(output)

        if self.res_attention: return output, attn_weights, attn_scores
        else: return output, attn_weights

In [ ]:
#| notest
mha_attn = MultiheadAttentionCustom(d_model=512, n_heads=8, attn_dropout=0., proj_dropout=0., res_attention=False)
mha_attn

MultiheadAttentionCustom(
  (W_Q): Linear(in_features=512, out_features=512, bias=True)
  (W_K): Linear(in_features=512, out_features=512, bias=True)
  (W_V): Linear(in_features=512, out_features=512, bias=True)
  (sdp_attn): ScaledDotProductAttention(
    (attn_dropout): Dropout(p=0.0, inplace=False)
  )
  (to_out): Sequential(
    (0): Linear(in_features=512, out_features=512, bias=True)
    (1): Dropout(p=0.0, inplace=False)
  )
)

In [ ]:
#| notest
def test_attention_equivalence():
    # Set random seed for reproducibility
    torch.manual_seed(42)
    
    # Test parameters
    batch_size = 2
    seq_len = 10
    d_model = 64
    n_heads = 4
    
    # Create input tensor (only need one since we're using self-attention)
    x = torch.randn(batch_size, seq_len, d_model)
    
    # Create key padding mask
    key_padding_mask = torch.zeros(batch_size, seq_len, dtype=torch.bool)
    key_padding_mask[:, -2:] = True  # mask last 2 positions

    # Initialize both implementations
    custom_mha = MultiheadAttentionCustom(d_model=d_model, n_heads=n_heads)
    flash_mha = MultiHeadAttention(d_model=d_model, n_heads=n_heads)
    
    # Set both models to eval mode to disable dropout
    custom_mha.eval()
    flash_mha.eval()
    
    # Copy weights to ensure identical parameters
    # Combine QKV weights from custom implementation into single matrix for flash attention
    combined_weight = torch.cat([
        custom_mha.W_Q.weight,
        custom_mha.W_K.weight,
        custom_mha.W_V.weight
    ], dim=0)
    combined_bias = torch.cat([
        custom_mha.W_Q.bias,
        custom_mha.W_K.bias,
        custom_mha.W_V.bias
    ], dim=0)
    
    # Copy combined weights to flash attention
    flash_mha.c_attn.weight.data = combined_weight
    flash_mha.c_attn.bias.data = combined_bias
    
    # Output projection weights
    flash_mha.c_proj.weight.data = custom_mha.to_out[0].weight.data.clone()
    flash_mha.c_proj.bias.data = custom_mha.to_out[0].bias.data.clone()
    
    # Forward pass
    with torch.no_grad():
        custom_output, custom_attn = custom_mha(x, key_padding_mask=key_padding_mask)
        
        flash_output = flash_mha(x, attn_mask=key_padding_mask)
    
    # Compare outputs
    print(f"Custom output shape: {custom_output.shape}")
    print(f"Flash output shape: {flash_output.shape}")
    
    output_close = torch.allclose(custom_output, flash_output, rtol=0, atol=0)
    print(f"Outputs match: {output_close}")
    
    if not output_close:
        print("\nOutput differences:")
        print(f"Max difference: {(custom_output - flash_output).abs().max().item()}")
        print(f"Mean difference: {(custom_output - flash_output).abs().mean().item()}")
    
    return custom_output, flash_output

custom_output, flash_output = test_attention_equivalence()
#: 8.940696716308594e-08
#Mean difference: 1.0550138540565968e-08

Custom output shape: torch.Size([2, 10, 64])
Flash output shape: torch.Size([2, 10, 64])
Outputs match: True


In [ ]:
#| notest
d_model=512
n_heads=8
d_k = d_v = d_model // n_heads
attn = ScaledDotProductAttention(d_model=d_model, n_heads=n_heads)
mha_attn = MultiheadAttentionCustom(d_model, n_heads)

W_Q = nn.Linear(d_model, d_k * n_heads)
W_K = nn.Linear(d_model, d_k * n_heads)
W_V = nn.Linear(d_model, d_v * n_heads)
X,_,_ = ds[0]

X = create_patch(X, patch_len=(10*50), stride=(5*50), constant_pad=True)

patch_len = X.shape[-1]

X = X[None, ...].permute(0,2,1,3)  # simulate batch size of 1 [bs x n_vars x num_patch x patch_len]

print(f'X input shape: {X.shape}')
W_P = nn.Linear(patch_len, d_model)

X = W_P(X) # project to d_model
print(f"Projected X shape to d_model: {X.shape}")

X = torch.reshape(X, (X.shape[0]*X.shape[1],X.shape[2],X.shape[3]))
print(f"Reshape for attention: {X.shape}")

# test multihead attention
print("\nTesting MHA and SDA attention, with just 50 elements.")
mha_output, mha_attn_weights = mha_attn(Q=X[:,:50,:])
print(f"MHA attention output shape: {mha_output.shape}, mha attn weight shape: {mha_attn_weights.shape}")

# test scaled dot product attn
K = Q = V = X

# # Linear (+ split in multiple heads)
bs = 1 # 1 * 16
q_s = W_Q(Q).reshape(bs, -1, n_heads, d_k).transpose(1, 2)
k_s = W_K(K).reshape(bs, -1, n_heads, d_k).permute(0, 2, 3, 1)
v_s = W_V(V).reshape(bs, -1, n_heads, d_v).transpose(1, 2)
print(f"Q shape: {q_s.shape}, K shape: {k_s.shape}, V shape: {v_s.shape}")

to_out = nn.Linear(n_heads * d_v, d_model)
output, attn_weights = attn(q_s[:,:,:50,:],k_s[:,:,:,:50], v_s[:,:,:50,:])
output = output.transpose(1, 2).contiguous().view(bs, -1, n_heads * d_v)
print(f"Attn output shape {output.shape}, attn weight shape: {attn_weights.shape}")

X input shape: torch.Size([1, 7, 10799, 500])
Projected X shape to d_model: torch.Size([1, 7, 10799, 512])
Reshape for attention: torch.Size([7, 10799, 512])

Testing MHA and SDA attention, with just 50 elements.
MHA attention output shape: torch.Size([7, 50, 512]), mha attn weight shape: torch.Size([7, 8, 50, 50])
Q shape: torch.Size([1, 8, 75593, 64]), K shape: torch.Size([1, 8, 64, 75593]), V shape: torch.Size([1, 8, 75593, 64])
Attn output shape torch.Size([1, 50, 512]), attn weight shape: torch.Size([1, 8, 50, 50])


In [ ]:
#| export
class Attention_Rel_Scl(nn.Module):
    def __init__(self, 
        d_model:int, # Embedding dimension
        n_heads:int, # number of attention heads
        seq_len:int, # sequence length or num patches
        d_k:int=None, # key dimension
        d_v:int=None, # value dimension
        res_attention:bool=False, # whether to use residual attention
        attn_dropout:float=0., # dropout for attention
        lsa:bool=False, # whether to use LSA, trainable paramater for scaling
        proj_dropout:float=0., # dropout for projection
        qkv_bias:bool=True, # bias for q, k, v
        ):
        """
        Adapted from tsai
        Added residual attention and dropout
        """
        super().__init__()

        self.seq_len = seq_len
        head_dim = d_model // n_heads
        self.scale = nn.Parameter(torch.tensor(head_dim ** -0.5), requires_grad=lsa)

        d_k = d_model // n_heads if d_k is None else d_k
        d_v = d_model // n_heads if d_v is None else d_v

        self.n_heads, self.d_k, self.d_v = n_heads, d_k, d_v

        self.query = nn.Linear(d_model, d_k * n_heads, bias=qkv_bias)
        self.key = nn.Linear(d_model, d_k * n_heads, bias=qkv_bias)
        self.value = nn.Linear(d_model, d_v * n_heads, bias=qkv_bias)

        self.res_attention = res_attention

        self.relative_bias_table = nn.Parameter(torch.zeros((2 * self.seq_len - 1), n_heads))
        coords = torch.meshgrid((torch.arange(1), torch.arange(self.seq_len)), indexing="xy")
        coords = torch.flatten(torch.stack(coords), 1)
        relative_coords = coords[:, :, None] - coords[:, None, :]
        relative_coords[1] += self.seq_len - 1
        relative_coords = relative_coords.permute(1, 2, 0)
        relative_index = relative_coords.sum(-1).flatten().unsqueeze(1)
        self.register_buffer("relative_index", relative_index)

        self.attn_dropout = nn.Dropout(attn_dropout)
        self.to_out = nn.Sequential(nn.Linear(n_heads * d_v, d_model), nn.Dropout(proj_dropout))

    def forward(self, x, prev:Optional[Tensor]=None, key_padding_mask:Optional[Tensor]=None, attn_mask:Optional[Tensor]=None):
        batch_size = x.shape[0]
        
        q = self.query(x).view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)
        k = self.key(x).view(batch_size, -1, self.n_heads, self.d_k).permute(0, 2, 3, 1)
        v = self.value(x).view(batch_size, -1, self.n_heads, self.d_v).transpose(1, 2)

        attn_scores = torch.matmul(q, k) * self.scale # [seq_len, seq_len]
        if prev is not None: attn_scores = attn_scores + prev

        if attn_mask is not None:                                     # attn_mask with shape [q_len x seq_len] - only used when q_len == seq_len
            if attn_mask.dtype == torch.bool:
                attn_scores.masked_fill_(attn_mask, -np.inf)
            else:
                attn_scores += attn_mask

        # Key padding mask (optional)
        if key_padding_mask is not None:# and torch.any(key_padding_mask): - not sure if this is needed                         # mask with shape [bs x q_len] (only when max_w_len == q_len)
            attn_scores.masked_fill_(key_padding_mask.unsqueeze(1).unsqueeze(2).to(torch.bool), -np.inf)
        
        attn_weights = F.softmax(attn_scores, dim=-1)

        relative_bias = self.relative_bias_table.gather(0, self.relative_index.repeat(1, self.n_heads))
        relative_bias = relative_bias.reshape(self.seq_len, self.seq_len, -1).permute(2, 0, 1).unsqueeze(0)
        attn_weights = attn_weights + relative_bias

        if attn_mask is not None and attn_mask.dtype == torch.bool:
            attn_weights = attn_weights.masked_fill(attn_mask, 0.0)
       
        if key_padding_mask is not None:# and torch.any(key_padding_mask):
            attn_weights = attn_weights.masked_fill(key_padding_mask.unsqueeze(1).unsqueeze(2).to(torch.bool), 0.0)

        attn_weights = self.attn_dropout(attn_weights)

        out = torch.matmul(attn_weights, v) # [batch_size, n_heads, seq_len, d_head]
        out = out.transpose(1, 2).contiguous().view(batch_size, -1, self.n_heads * self.d_v) # output: [bs x q_len x n_heads * d_v]
        out = self.to_out(out)
        if self.res_attention: 
            return out, attn_weights, attn_scores
        else: 
            return out, attn_weights

In [ ]:
#| notest
## test w patches [bs *c_in x num_patches x d_model]
d_model=512
c_in = 2
num_patches = 10
x_emb = torch.randn(4*c_in,num_patches, d_model)
abs_position = tAPE(d_model, seq_len=num_patches)
x_emb_pos = abs_position(x_emb)

model = Attention_Rel_Scl(d_model=d_model,
        n_heads=2, # number of attention heads
        seq_len=num_patches, # sequence length or num patches
        )

out, attn_weights = model(x_emb)


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()